# Reflexion Architecture — CrewAI

**Pattern:** Generator → Critic → Rewriter → `ReflexionOutput`

```
generator → critique_task(context=[t1]) → rewrite_task(context=[t1,t2]) → ReflexionOutput
```

CrewAI models reflexion as a 3-task pipeline. The rewriter always runs — critic's feedback is always applied.

In [ ]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0)
print("✓ Ready")

In [ ]:
class ReflexionOutput(BaseModel):
    final_recommendation: str
    quality_score: int = Field(ge=1, le=10)
    improvements_made: str

generator = Agent(role="Travel Recommendation Writer",
    goal="Write comprehensive recommendations with attractions, weather, safety, best time.",
    backstory="Detail-oriented travel writer.", tools=[], llm=gemini, verbose=False)
critic = Agent(role="Travel Content Editor",
    goal="Score and critique recommendations rigorously.",
    backstory="High-standard editor who always finds gaps.", tools=[], llm=gemini, verbose=False)
rewriter = Agent(role="Travel Recommendation Rewriter",
    goal="Polish drafts by fixing every identified gap.",
    backstory="Takes rough drafts and makes them excellent.", tools=[], llm=gemini, verbose=False)
print("3 reflexion agents ready")

In [ ]:
destination = "Tokyo"

t1 = Task(description=f"Write a travel recommendation for {destination}: top 3 attractions, weather, safety, best time.",
    expected_output="Comprehensive recommendation.", agent=generator)
t2 = Task(description=f"Critique the {destination} recommendation. Score 1-10 on specificity, weather, safety, detail. List all gaps.",
    expected_output="Score and gap list.", agent=critic, context=[t1])
t3 = Task(description=f"Rewrite the {destination} recommendation addressing ALL critic feedback. Fill ReflexionOutput.",
    expected_output="A complete ReflexionOutput.", agent=rewriter, context=[t1,t2], output_pydantic=ReflexionOutput)

result = Crew(agents=[generator,critic,rewriter], tasks=[t1,t2,t3], process=Process.sequential, verbose=False).kickoff()
output: ReflexionOutput = result.pydantic
if output:
    print(f"Quality score: {output.quality_score}/10")
    print(f"Improvements: {output.improvements_made}")
    print(f"\nFinal Recommendation:\n{output.final_recommendation}")
else:
    print(result.raw)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Rewriter always runs | CrewAI reflexion is deterministic — always generate+critique+rewrite |
| `context=[t1,t2]` on rewriter | Rewriter sees original AND critique side by side |
| `ReflexionOutput.improvements_made` | Captures what changed — useful for audit trails |

**CrewAI vs LangGraph reflexion:**  
CrewAI always runs the rewriter (fixed 3-task pipeline). LangGraph can skip rewrite if score passes (conditional edge). LangGraph is more efficient but requires more setup.

### Next: [ADK Reflexion →](../ADK/reflexion.ipynb)